# 15 — Bag of Words (BoW)

Bag of Words turns text into a fixed-size numeric vector: count how often each vocabulary term appears, and that vector is the document. The order of words is discarded — hence "bag". Simple, fast, and the foundation every later representation (TF-IDF, embeddings) builds on.

**Why it matters for resumes / ATS:** BoW is the baseline every matching engine starts from. A resume and a job description can be compared as vectors; skills that appear in both push the vectors closer. Understanding BoW's strengths (cheap, exact) and limits (no order, no semantics) tells you when to use it and when to reach for Ch. 16 and beyond.

![Text Representation Evolution](../../../assets/images/text_representation_evolution_1785491155497.png)

> **Figure:** Evolution of text representation methods. Bag of Words is the starting point — Blocks C and D cover each level in sequence.

The figure places BoW at the bottom of the representation ladder: counts first, then TF-IDF weighting (Ch. 16), n-grams (Ch. 17), and dense embeddings (Ch. 19–22). Each level fixes a weakness of the one below — keep this map in mind as the chapters progress.

**Goal:** Convert text into numerical feature vectors using CountVectorizer.

Vectorizing is the prerequisite for any math on text: similarity, ranking, and classification all need numbers. After this chapter you can build a document-term matrix with `CountVectorizer` and read it like a table — rows are documents, columns are vocabulary terms, cells are counts.

## 1. How BoW Works

`CountVectorizer` does two jobs: builds the vocabulary (unique terms across all documents) and counts occurrences per document into a matrix. Each cell is a raw term frequency.

**What the code does:** fits on three short docs and prints the vocabulary, matrix shape, and the first document's row.
- Vocabulary: 9 unique terms — `['for', 'fun', 'great', 'is', 'learning', 'machine', 'nlp', 'python', 'with']` (lowercased, punctuation stripped).
- Shape `(3, 9)`: 3 documents × 9 vocabulary terms; doc 1 ("Python is great for NLP") is `[1, 0, 1, 1, 0, 0, 1, 1, 0]` — 1 where the term appears, 0 elsewhere.

**Try it:** count by hand — "python", "is", "great", "for", "nlp" each appear once, so doc 1's row has five 1s. The position of the 1s is meaningless; only the counts matter.

In [1]:
from sklearn.feature_extraction.text import CountVectorizer
docs = [
    "Python is great for NLP",
    "NLP with Python is fun",
    "Machine learning with Python",
]
vec = CountVectorizer()
matrix = vec.fit_transform(docs)
print(f"Vocabulary: {vec.get_feature_names_out()}")
print(f"Matrix shape: {matrix.shape}")
print(f"First doc: {matrix[0].toarray()[0]}")

Vocabulary: ['for' 'fun' 'great' 'is' 'learning' 'machine' 'nlp' 'python' 'with']
Matrix shape: (3, 9)
First doc: [1 0 1 1 0 0 1 1 0]


## 2. Vocabulary Size vs Sparsity

Real text vectors are mostly zeros: a 10,000-term vocabulary, a 300-word resume, so ~97% of the matrix is empty. Sparsity is the number to watch — it drives memory and speed.

**What the code does:** re-fits `CountVectorizer` with `max_features` 5→50 and prints vocab size and sparsity `1 - nnz/(rows*cols)`.
- On this toy corpus the vocabulary saturates at 6 terms and sparsity at `0.500` — modest, because 3 tiny documents can't fill a large vocabulary.

**Try it:** the code's printout claims ">90% zeros", but the measured sparsity here is only ~50%. That claim is true for real corpora (thousands of docs, thousands of terms) — this toy set is too small to show it. Scale up the document list and watch sparsity climb.

In [4]:
import numpy as np
for max_f in [5, 10, 20, 50]:
    v = CountVectorizer(max_features=max_f, stop_words="english")
    m = v.fit_transform(docs)
    sparsity = 1 - m.nnz / (m.shape[0] * m.shape[1])
    print(f"max_features={max_f:3d}: vocab={len(v.get_feature_names_out()):3d}, sparsity={sparsity:.3f}")

print("\nKey insight: BoW produces highly sparse vectors (>90% zeros).")

max_features=  5: vocab=  5, sparsity=0.467
max_features= 10: vocab=  6, sparsity=0.500
max_features= 20: vocab=  6, sparsity=0.500
max_features= 50: vocab=  6, sparsity=0.500

Key insight: BoW produces highly sparse vectors (>90% zeros).


## 3. BoW for Resume Comparison

With `binary=True` each cell records *presence* (1/0) instead of count — a resume says "TensorFlow" once or ten times; either way the candidate knows it. Binary vectors make overlap counting trivial: element-wise products sum to the number of shared terms.

**What the code does:** vectorizes three resumes into a 3×14 binary matrix and computes shared-term counts between resume 1 and the others.
- Resume 1 ("Data scientist Python TensorFlow...") and resume 2 ("Python developer Django Flask...") share **1** word (`python`).
- Resume 1 and resume 3 ("ML engineer TensorFlow PyTorch...") share **2** words (`tensorflow`, `learning`).

**Try it:** read the printed 0/1 matrix — resume 1's row has 1s at `data, learning, machine, python, scientist, tensorflow`, which is exactly the set the sharing counts come from.

In [5]:
resumes = [
    "Data scientist Python TensorFlow machine learning",
    "Python developer Django Flask backend engineer",
    "ML engineer TensorFlow PyTorch deep learning",
]
vec = CountVectorizer(binary=True, stop_words="english")
X = vec.fit_transform(resumes)
print("Resume-Keyword matrix (binary):")
print(vec.get_feature_names_out())
print(X.toarray())
print(f"\nResume 1 and 2 share {sum(X[0].multiply(X[1]).toarray()[0])} words")
print(f"Resume 1 and 3 share {sum(X[0].multiply(X[2]).toarray()[0])} words")

Resume-Keyword matrix (binary):
['backend' 'data' 'deep' 'developer' 'django' 'engineer' 'flask'
 'learning' 'machine' 'ml' 'python' 'pytorch' 'scientist' 'tensorflow']
[[0 1 0 0 0 0 0 1 1 0 1 0 1 1]
 [1 0 0 1 1 1 1 0 0 0 1 0 0 0]
 [0 0 1 0 0 1 0 1 0 1 0 1 0 1]]

Resume 1 and 2 share 1 words
Resume 1 and 3 share 2 words


## Key Insight: BoW is simple but loses word order and semantics. Use as a baseline.

**BoW is the honest baseline: cheap, exact, and blind — it sees "not skilled in Python" as a Python hit.**

The bag discards word order ("Python data science" and "data science Python" are identical) and any notion of meaning (synonyms never match). That's why every later chapter fixes a specific blind spot: TF-IDF weights terms (Ch. 16), n-grams restore local order (Ch. 17), and embeddings restore semantics (Ch. 19+). Keep BoW as your benchmark — any fancier method must beat it on the same metric.